# AWS Identity and Access Management (IAM)

IAM is the **security foundation** of every AWS workload. It answers: *who can do what, on which resource, under what conditions.*

## Core Concepts

| Entity | Description |
|---|---|
| **User** | Long-lived identity for a person or application (avoid for services) |
| **Group** | Collection of users that share a set of permissions |
| **Role** | Assumable identity — no long-lived credentials; preferred for services |
| **Policy** | JSON document defining permissions (Allow / Deny on Actions + Resources) |
| **STS** | Security Token Service — issues short-lived credentials when a role is assumed |

### Principle of Least Privilege
Grant only the permissions required to perform the task — nothing more.
Start with zero permissions and add incrementally.

---

### Prerequisites for Students

Before running this notebook, ensure you have:
1. Installed **boto3**:
   ```bash
   pip install boto3
   ```
2. Configured your credentials in terminal:
   ```bash
   aws configure
   ```
   *(Enter your AWS Access Key ID, Secret Access Key, and default region `us-east-1`)*
3. Verified authentication:
   ```bash
   aws sts get-caller-identity
   ```


In [2]:
import os
import json
import boto3

# Initialize Boto3 session:
# Automatically uses AWS_PROFILE if set in environment, otherwise uses default credentials
aws_profile = os.environ.get('AWS_PROFILE', "default")
session = boto3.Session(profile_name=aws_profile)
iam = session.client('iam')
sts = session.client('sts')

# Confirm the identity you are operating as before making changes
identity = sts.get_caller_identity()
username = identity['Arn'].split('/')[-1]

print(f'Account  : {identity["Account"]}')
print(f'UserID   : {identity["UserId"]}')
print(f'User ARN : {identity["Arn"]}')
print(f'Username : {username}')


Account  : 281466184825
UserID   : AIDAUDCFRER43NAR4HJVG
User ARN : arn:aws:iam::281466184825:user/cde-ole
Username : cde-ole


In [ ]:
# ── List existing IAM entities ────────────────────────────────────────────────
users = iam.list_users()['Users']
print(f'Users ({len(users)}):')
for u in users[:5]:
    created = u['CreateDate'].strftime('%Y-%m-%d')
    print(f'  {u["UserName"]:35s}  created: {created}')

roles = iam.list_roles()['Roles']
print(f'\nRoles ({len(roles)}) — first 5 AWS-managed excluded:')
custom = [r for r in roles if not r['Path'].startswith('/aws-service-role/')]
for r in custom[:5]:
    print(f'  {r["RoleName"]:45s}  {r["Arn"]}')

Users (2):
  cde-ole                              created: 2026-10-05
  cde-team-ole                         created: 2026-10-05

Roles (3) — first 5 AWS-managed excluded:


## Creating a Least-Privilege Policy

A policy document has three key fields per statement:
- **Effect**: `Allow` or `Deny`
- **Action**: the API operations being permitted (e.g. `s3:GetObject`)
- **Resource**: the ARN(s) the actions apply to (use `*` only when unavoidable)

In [10]:
# Suffix resource names with username to prevent collisions across students in shared accounts
POLICY_NAME = f'ClassicModelsETLReadOnly-{username}'

policy_doc = {
    'Version': '2012-10-17',
    'Statement': [
        {
            'Sid': 'ReadS3DataLake',
            'Effect': 'Allow',
            'Action': ['s3:GetObject', 's3:ListBucket', 's3:GetBucketLocation'],
            'Resource': [
                f'arn:aws:s3:::classicmodels-data-lake-{username}',
                f'arn:aws:s3:::classicmodels-data-lake-{username}/*'
            ]
        },
        {
            'Sid': 'DescribeRDSReadOnly',
            'Effect': 'Allow',
            'Action': [
                'rds:DescribeDBInstances',
                'rds:DescribeDBClusters',
                'rds:ListTagsForResource'
            ],
            'Resource': '*'
        },
        {
            'Sid': 'DenyDeleteOperations',
            'Effect': 'Deny',
            'Action': ['s3:DeleteObject', 's3:DeleteBucket', 'rds:DeleteDBInstance'],
            'Resource': '*'
        }
    ]
}

try:
    response = iam.create_policy(
        PolicyName=POLICY_NAME,
        PolicyDocument=json.dumps(policy_doc),
        Description=f'Read-only access to ClassicModels data lake and RDS for {username}'
    )
    policy_arn = response['Policy']['Arn']
    print(f'Policy created: {policy_arn}')
except iam.exceptions.EntityAlreadyExistsException:
    policy_arn = f"arn:aws:iam::{identity['Account']}:policy/{POLICY_NAME}"
    print(f'Policy already exists, using: {policy_arn}')


Policy created: arn:aws:iam::673868928338:policy/ClassicModelsETLReadOnly-ashish


## Creating a Service Role

Roles are the right way to grant permissions to AWS services (EC2, Lambda, Glue).
A **trust policy** defines which principal is allowed to assume the role.

In [11]:
ROLE_NAME = f'ClassicModelsETLRole-{username}'

# Trust policy: allow EC2 instances AND the current student identity to assume this role
trust_policy = {
    'Version': '2012-10-17',
    'Statement': [{
        'Effect': 'Allow',
        'Principal': {
            'Service': 'ec2.amazonaws.com',
            'AWS': [
                identity['Arn'],
                f"arn:aws:iam::{identity['Account']}:root"
            ]
        },
        'Action': 'sts:AssumeRole'
    }]
}

try:
    role = iam.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description=f'ETL pipeline role for ClassicModels - {username}',
        MaxSessionDuration=3600   # 1-hour sessions
    )
    role_arn = role['Role']['Arn']
    print(f'Role created: {role_arn}')
except iam.exceptions.EntityAlreadyExistsException:
    role_arn = iam.get_role(RoleName=ROLE_NAME)['Role']['Arn']
    # Ensure trust policy on existing role allows current student to assume it
    iam.update_assume_role_policy(
        RoleName=ROLE_NAME,
        PolicyDocument=json.dumps(trust_policy)
    )
    print(f'Role already exists — updated trust policy for {identity["Arn"]}: {role_arn}')

# Attach the least-privilege policy to the role
if 'policy_arn' in locals():
    try:
        iam.attach_role_policy(RoleName=ROLE_NAME, PolicyArn=policy_arn)
        print(f'Policy attached: {POLICY_NAME}')
    except Exception as e:
        print(f'Could not attach policy: {e}')

# Verify attached policies
attached = iam.list_attached_role_policies(RoleName=ROLE_NAME)['AttachedPolicies']
for p in attached:
    print(f'  ✓ {p["PolicyName"]}')


Role created: arn:aws:iam::673868928338:role/ClassicModelsETLRole-ashish
Policy attached: ClassicModelsETLReadOnly-ashish
  ✓ ClassicModelsETLReadOnly-ashish


## (Optional / Conceptual) How STS Issues Temporary Credentials

In production data engineering, **you almost never write Python code to assume a role manually**.
- AWS services (like **EC2, AWS Glue, AWS Lambda, EMR, and MWAA/Airflow**) assume service roles **automatically** in the background when running your pipeline.
- The cell below is an **optional peek behind the scenes** to show how AWS Security Token Service (STS) issues short-lived, auto-expiring credentials (`AccessKeyId`, `SecretAccessKey`, `SessionToken`).


In [5]:
# ── (Optional) STS Under the Hood ─────────────────────────────────────────────
# In production, AWS services assume roles automatically.
# Here we test assuming the role programmatically to see temporary credentials in action.

if 'role_arn' not in locals():
    print('Notice: role_arn not found. Please run Cell 6 first.')
else:
    # Attempt to auto-grant AssumeRole if user has IAM write permissions
    USER_POLICY_NAME = 'LabAllowAssumeRole'
    try:
        iam.put_user_policy(
            UserName=username,
            PolicyName=USER_POLICY_NAME,
            PolicyDocument=json.dumps({
                'Version': '2012-10-17',
                'Statement': [{
                    'Effect': 'Allow',
                    'Action': 'sts:AssumeRole',
                    'Resource': role_arn
                }]
            })
        )
    except Exception:
        pass  # If user lacks permission to modify policies, proceed to try sts directly

    try:
        assumed = sts.assume_role(
            RoleArn=role_arn,
            RoleSessionName=f'ETLPipelineRun-{username}',
            DurationSeconds=900    # 15-minute temporary session
        )
        creds = assumed['Credentials']

        print('✓ Short-lived credentials successfully issued by STS:')
        print(f'  AccessKeyId   : {creds["AccessKeyId"][:12]}...')
        print(f'  Expiration    : {creds["Expiration"]}')
        print(f'  Session token : {creds["SessionToken"][:20]}...')

        # Use the temporary credentials to create a scoped S3 client
        s3_scoped = boto3.client(
            's3',
            aws_access_key_id=creds['AccessKeyId'],
            aws_secret_access_key=creds['SecretAccessKey'],
            aws_session_token=creds['SessionToken']
        )
        print('\nScoped S3 client ready — credentials expire automatically after 15 minutes.')

    except Exception as e:
        print(f'Notice (Expected in restricted student accounts): sts.assume_role skipped.')
        print(f'Reason: {e}')
        print('\n[Key Takeaway for Students]:')
        print('In production, data services (EC2/Glue/Lambda) assume service roles automatically.')
        print('Manual client-side sts.assume_role() is not required for daily data engineering tasks.')


✓ Short-lived credentials successfully issued by STS:
  AccessKeyId   : ASIAZZZNX2VJ...
  Expiration    : 2026-10-05 18:17:37+00:00
  Session token : IQoJb3JpZ2luX2VjEBoa...

Scoped S3 client ready — credentials expire automatically after 15 minutes.


## Cleanup Lab Resources

> ⚠️ **Important Note for Students**:
> Only run the cleanup cell below **after** you have inspected your newly created policy and role in the [AWS IAM Console](https://console.aws.amazon.com/iam).
> Running this cell will detach and permanently delete the role and policy created in this lab.


In [6]:
# ── Safe Cleanup ──────────────────────────────────────────────────────────────
# Clean up lab resources to avoid unintended access and cost
if 'username' in locals() and 'USER_POLICY_NAME' in locals():
    try:
        iam.delete_user_policy(UserName=username, PolicyName=USER_POLICY_NAME)
        print(f'Deleted temporary user policy {USER_POLICY_NAME} from {username}')
    except Exception as e:
        pass

if 'policy_arn' in locals() and 'ROLE_NAME' in locals():
    try:
        iam.detach_role_policy(RoleName=ROLE_NAME, PolicyArn=policy_arn)
        print(f'Detached policy {POLICY_NAME} from role: {ROLE_NAME}')
    except Exception as e:
        print(f'Could not detach policy: {e}')

if 'ROLE_NAME' in locals():
    try:
        iam.delete_role(RoleName=ROLE_NAME)
        print(f'Deleted role: {ROLE_NAME}')
    except Exception as e:
        print(f'Could not delete role: {e}')

if 'policy_arn' in locals():
    try:
        iam.delete_policy(PolicyArn=policy_arn)
        print(f'Deleted policy: {policy_arn}')
    except Exception as e:
        print(f'Could not delete policy: {e}')


Deleted temporary user policy LabAllowAssumeRole from ashish
Detached policy ClassicModelsETLReadOnly-ashish from role: ClassicModelsETLRole-ashish
Deleted role: ClassicModelsETLRole-ashish
Deleted policy: arn:aws:iam::673868928338:policy/ClassicModelsETLReadOnly-ashish


## IAM Best Practices

| Practice | Why |
|---|---|
| Never use the root account for daily work | Root has unrestricted access; compromise is catastrophic |
| Use roles for services, not IAM users | Roles have no long-lived credentials to leak |
| Apply least-privilege from day one | Easier to add permissions than to remove them after an incident |
| Enable MFA for all human users | Protects against credential theft |
| Rotate access keys regularly | Limit the window of exposure for any leaked key |
| Use permission boundaries for delegation | Prevent privilege escalation when creating sub-roles |
| Enable CloudTrail | Audit every API call made in your account |
| Tag all IAM resources | Enables cost attribution and governance automation |